# Energy Consumption Forecasting Using Time-Series ML
## Exploratory Data Analysis & Feature Engineering
**Programme:** HCLTech Career Shaper Programme — III Year Industry Project  
**Project No.:** 33 | **Domain:** Digital-Data-AI / AIML  
**Author:** Rahul Kumar Verma  

---
### Objective
This notebook conducts in-depth exploratory data analysis (EDA) on building energy consumption data (`dataset.csv`).
It examines:
1. Data integrity, schemas, and missing value patterns
2. Consumption time series across anonymized facilities (`DataCenter`, `Office_A`, `Office_B`, `Warehouse`, `Canteen`)
3. Diurnal (24-hour) and weekly operational load cycles
4. Exogenous weather influences (temperature and humidity)
5. Anomaly boundaries and stationarity properties for time-series modeling

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 120

DATA_PATH = '../data/processed/dataset_augmented_5000.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../dataset.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset shape: {df.shape}")
df.head()

### 1. Data Schema & Cadence Verification
Let us examine data types, missing value counts, and temporal coverage.

In [ ]:
df['parsed_dt'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['parsed_dt'].dt.hour
df['day_name'] = df['parsed_dt'].dt.day_name()
df['is_weekend'] = df['parsed_dt'].dt.weekday >= 5

print("=== Summary Statistics ===")
display(df.describe())

print("\n=== Building Distribution ===")
print(df['building_id'].value_counts())

print("\n=== Day Type Counts ===")
print(df['day_type'].value_counts())

### 2. Visualization 1: Energy Consumption Over Time
Examining the chronological time series per building category reveals steady baseline loads versus fluctuating diurnal peaks.

In [ ]:
plt.figure(figsize=(14, 6))
palette = {'DataCenter': '#2563EB', 'Office_A': '#10B981', 'Office_B': '#F59E0B', 'Warehouse': '#8B5CF6', 'Canteen': '#EC4899'}

for b_id, color in palette.items():
    sub = df[df['building_id'] == b_id].sort_values('parsed_dt').iloc[:168]  # First 168 hours (1 full week)
    plt.plot(sub['parsed_dt'], sub['consumption_kWh'], label=b_id, color=color, linewidth=1.8, alpha=0.85)

plt.title("Figure 1: Chronological Hourly Energy Consumption Across 1 Week (168 Hours)", fontsize=13, fontweight='bold')
plt.xlabel("Observation Timestamp", fontsize=11)
plt.ylabel("Active Consumption (kWh)", fontsize=11)
plt.legend(title="Building Category", loc='upper right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Interpretation 1:**
- **DataCenter** exhibits the highest continuous base load (~50-70 kWh), characteristic of uninterruptible server infrastructure and chiller plants operating 24/7.
- **Office_A & Office_B** show pronounced diurnal peaks during business hours (08:00 - 18:00) and drop to low night base loads (~3-6 kWh).
- **Canteen** displays multi-modal peaks aligned with meal preparation schedules (breakfast, lunch, dinner).
- **Warehouse** demonstrates uniform daytime shift operations.

### 3. Visualization 2: Statistical Distribution & Outlier Detection
Boxplots and violin distributions highlighting variance, interquartile ranges (IQR), and potential anomaly regimes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

sns.boxplot(data=df, x='building_id', y='consumption_kWh', palette='Set2', ax=ax1)
ax1.set_title("Consumption Spread by Building (Boxplot)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Building ID", fontsize=10)
ax1.set_ylabel("Consumption (kWh)", fontsize=10)

sns.kdeplot(data=df, x='consumption_kWh', hue='building_id', common_norm=False, palette='Set2', ax=ax2, fill=True, alpha=0.3)
ax2.set_title("Empirical Density Function (KDE)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Consumption (kWh)", fontsize=10)

plt.tight_layout()
plt.show()

**Interpretation 2:**
- Facility consumption distributions are multi-modal across buildings, verifying the mentor instruction that each building must be modeled separately rather than aggregated into an incoherent single series.
- Outliers outside 1.5 * IQR correspond to high-surge events or abnormal sensor dropouts injected as anomaly benchmarks.

### 4. Visualization 3: Diurnal 24-Hour Cycle and Operating Schedules
Evaluating average load across the 24 hours of the day and comparing Weekday vs Weekend vs Holiday regimens.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Hour of day mean curve
hourly_curve = df.groupby(['hour', 'building_id'])['consumption_kWh'].mean().unstack()
hourly_curve.plot(ax=ax1, marker='o', linewidth=2)
ax1.set_title("Diurnal Profile Across 24 Hours", fontsize=12, fontweight='bold')
ax1.set_xlabel("Hour of Day (0-23)", fontsize=10)
ax1.set_ylabel("Mean Consumption (kWh)", fontsize=10)
ax1.set_xticks(range(0, 24, 2))

# Schedule comparison
schedule_curve = df.groupby(['day_type', 'building_id'])['consumption_kWh'].mean().unstack()
schedule_curve.plot(kind='bar', ax=ax2, alpha=0.85, edgecolor='black')
ax2.set_title("Mean Consumption: Weekday vs Weekend vs Holiday", fontsize=12, fontweight='bold')
ax2.set_xlabel("Operating Schedule Type", fontsize=10)
ax2.set_ylabel("Mean Consumption (kWh)", fontsize=10)
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

**Interpretation 3:**
- Strong daily seasonality ($m=24$) confirms the necessity of seasonal lag structures in SARIMA and LSTM recurrent sequence windows ($W \ge 24$).
- Significant drops on Weekends and Holidays for offices highlight the importance of calendar exogenous variables (`is_weekend`, `day_type`).

### 5. Visualization 4: Exogenous Thermal Correlation Matrix

In [ ]:
plt.figure(figsize=(7, 5.5))
num_cols = ['consumption_kWh', 'temperature_C', 'humidity_pct', 'hour']
corr = df[num_cols].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", cbar_kws={'label': 'Correlation'})
plt.title("Correlation with Exogenous Temperature & Humidity", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Conclusion & Modeling Implications
1. **Cadence:** Hourly observations require a 24-step daily seasonal cycle and 168-step weekly cycle.
2. **Decomposition:** Additive seasonality is appropriate for baseline building models, with exogenous temperature contributing positively to cooling loads.
3. **Model Selection:**
   - **Baseline:** Seasonal Naive (24-hour lag)
   - **Statistical:** SARIMA(1,1,1)x(1,1,1)24
   - **Deep Learning:** LSTM (24-hour input sequence, 1-step ahead recursive forecasting)
   - **Additive:** Prophet / Random Forest regressor with lag features
4. **Evaluation:** Evaluated on strict chronological holdout test set (no random shuffling, zero future leakage) with MAE, RMSE, and safe MAPE.